In [2]:
!pip install google-cloud-vision unidecode tqdm joblib -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.3 MB/s eta 0:00:00


In [3]:
import os
from google.cloud import vision
import io

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value = user_secrets.get_secret("VISION_API_CREDENTIALS")
    
    with open("kaggle_secret.json", "w") as f:
        f.write(secret_value)
        
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "kaggle_secret.json"
    print("✅ Đã thiết lập thông tin xác thực Google Cloud thành công!")

except Exception as e:
    print(f"❌ Lỗi khi thiết lập xác thực: {e}")

✅ Đã thiết lập thông tin xác thực Google Cloud thành công!


In [4]:
import os
import cv2
import json
import numpy as np
import pickle  # <-- THAY ĐỔI: Thêm thư viện pickle
import time    # <-- THAY ĐỔI: Thêm thư viện time để chạy được logic retry
from google.cloud import vision
# <-- THAY ĐỔI: Thêm import cho các exception của Google API
from google.api_core.exceptions import ResourceExhausted, InternalServerError, ServiceUnavailable
from unidecode import unidecode
from tqdm.notebook import tqdm
from joblib import Parallel, delayed
from typing import List, Tuple, Optional, Dict
from collections import defaultdict

class ExtractText:
    """
    Pipeline trích xuất và chuẩn hóa văn bản từ một danh sách ảnh,
    sử dụng Google Vision API và cho phép xử lý một phần dữ liệu (slicing).
    """

    def __init__(self, paths_to_process_file: str, output_path: str, progress: Optional[Dict[str, int]] = None):
        """
        Khởi tạo pipeline.

        Args:
            paths_to_process_file (str): Đường dẫn đến file .pkl chứa list các đường dẫn ảnh cần xử lý.
            output_path (str): Đường dẫn để lưu các file JSON kết quả.
            progress (Optional[Dict[str, int]]): Dictionary để kiểm soát phạm vi xử lý.
                Ví dụ: {'start': 0, 'end': 1000}.
                Nếu không được cung cấp, sẽ xử lý toàn bộ ảnh từ file.
        """
        self.output_path = output_path
        self.vision_client = vision.ImageAnnotatorClient()
        
        os.makedirs(self.output_path, exist_ok=True)
        print(f"Đọc danh sách ảnh từ file: {paths_to_process_file}")
        print(f"Kết quả JSON sẽ được lưu tại: {self.output_path}")

        # 1. Đọc file pkl chứa danh sách các đường dẫn ảnh
        print("Đang đọc và sắp xếp các đường dẫn ảnh từ file pkl...")
        try:
            with open(paths_to_process_file, 'rb') as f:
                all_image_paths = pickle.load(f)
        except FileNotFoundError:
            raise FileNotFoundError(f"File pkl không được tìm thấy tại: {paths_to_process_file}")
        except Exception as e:
            raise ValueError(f"Lỗi khi đọc hoặc giải nén file pkl '{paths_to_process_file}': {e}")

        if not isinstance(all_image_paths, list):
            raise TypeError(f"Dữ liệu trong file pkl phải là một list, nhưng lại là {type(all_image_paths)}.")
        
        all_image_paths.sort() 
        
        # 2. Xác định phạm vi xử lý (slicing) dựa trên progress
        total_images = len(all_image_paths)
        self.start_index = 0
        self.end_index = total_images

        # Nếu progress được cung cấp, cập nhật lại start và end
        if progress:
            self.start_index = progress.get('start', 0)
            self.end_index = progress.get('end', total_images) 
        
        print({
            "length": total_images,
            "start": self.start_index,
            "end": self.end_index
        })

        # Đảm bảo các chỉ số nằm trong giới hạn hợp lệ
        self.start_index = max(0, self.start_index)
        self.end_index = min(total_images, self.end_index)

        # Cắt danh sách paths để lấy phần cần xử lý
        if self.start_index >= self.end_index:
             print(f"Cảnh báo: 'start' ({self.start_index}) lớn hơn hoặc bằng 'end' ({self.end_index}). Sẽ không có ảnh nào được xử lý.")
             self.paths_to_process = []
        else:
            self.paths_to_process = all_image_paths[self.start_index:self.end_index]

        print(f"Đã tìm thấy tổng cộng {total_images} ảnh trong file pkl.")
        print(f"Sẽ xử lý {len(self.paths_to_process)} ảnh từ chỉ số {self.start_index} đến {self.end_index-1}.")

    def _apply_upscale_to_image(self, image: np.ndarray, scale_factor: float) -> Optional[np.ndarray]:
        if image is None: return None
        width = int(image.shape[1] * scale_factor)
        height = int(image.shape[0] * scale_factor)
        return cv2.resize(image, (width, height), interpolation=cv2.INTER_CUBIC)

    
    @staticmethod
    def _read_image_for_api(path: str, scale_factor = 2.0) -> Optional[Tuple[bytes, str]]:
        """Đọc ảnh, scale bằng hàm nội bộ, và mã hóa ảnh để gửi cho Vision API."""
        try:
            image = cv2.imread(path)
            if image is None:
                raise ValueError(f"Không thể đọc ảnh từ {path}")

            # Resize ảnh bằng hàm đã có
            image = self._apply_upscale_to_image(image, scale_factor)
            if image is None:
                raise ValueError(f"Lỗi khi upscale ảnh từ {path}")

            # Encode ảnh thành JPEG bytes
            success, encoded_image = cv2.imencode('.jpg', image)
            if not success:
                raise ValueError(f"Không thể encode ảnh từ {path}")

            return (encoded_image.tobytes(), path)
        except Exception as e:
            print(f"Lỗi khi xử lý ảnh {path}: {e}")
            return None

    @staticmethod
    def _normalize_text(raw_text: str) -> str:
        """Chuẩn hóa văn bản: gộp dòng, viết thường, bỏ dấu."""
        if not raw_text:
            return ""
        single_line = raw_text.replace('\n', ' ').strip()
        normalized_text = unidecode(single_line.lower())
        return normalized_text

    def _process_batch_with_google_api(self, image_batch: List[Tuple[bytes, str]]) -> List[Tuple[str, str]]:
        """Gửi một batch ảnh đến Vision API và trả về kết quả, với cơ chế thử lại."""
        requests = []
        original_paths = []
        for content, path in image_batch:
            image = vision.Image(content=content)
            features = [vision.Feature(type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION)]
            requests.append(vision.AnnotateImageRequest(image=image, features=features))
            original_paths.append(path)

        max_retries = 5
        base_delay = 1

        for attempt in range(max_retries):
            try:
                response = self.vision_client.batch_annotate_images(requests=requests)
                
                results = []
                for i, res in enumerate(response.responses):
                    if res.error.message:
                        print(f"Lỗi API cho ảnh {original_paths[i]}: {res.error.message}")
                        results.append(("", original_paths[i]))
                        continue
                    
                    if res.full_text_annotation:
                        results.append((res.full_text_annotation.text, original_paths[i]))
                    else:
                        results.append(("", original_paths[i]))
                return results

            except (ResourceExhausted, InternalServerError, ServiceUnavailable) as e:
                print(f"Gặp lỗi '{type(e).__name__}' ở lần thử {attempt + 1}/{max_retries}. Đang thử lại...")
                if attempt < max_retries - 1:
                    delay = base_delay * (2 ** attempt)
                    print(f" -> Sẽ chờ {delay} giây...")
                    time.sleep(delay)
                else:
                    print(f"Lỗi nghiêm trọng sau {max_retries} lần thử lại: {e}")
                    return [("", path) for _, path in image_batch]
            except Exception as e:
                print(f"Lỗi nghiêm trọng không thể thử lại khi gọi batch API: {e}")
                return [("", path) for _, path in image_batch]
        
        return [("", path) for _, path in image_batch]

    def extract_and_save(self, batch_size: int = 16):
        """
        Trích xuất văn bản từ các ảnh đã được chọn theo batch và lưu kết quả.
        
        Args:
            batch_size (int): Số lượng ảnh gửi đến API trong một lần gọi (tối đa 16).
        """
        if not self.paths_to_process:
            print("Không có ảnh nào để xử lý. Kết thúc.")
            return
            
        if batch_size > 16:
            print(f"Cảnh báo: Batch size ({batch_size}) lớn hơn giới hạn của Vision API (16). Đã giảm xuống 16.")
            batch_size = 16
            
        all_results = defaultdict(dict)
        
        with tqdm(total=len(self.paths_to_process), desc="Đang trích xuất văn bản") as pbar:
            for i in range(0, len(self.paths_to_process), batch_size):
                path_batch = self.paths_to_process[i:i+batch_size]
                
                image_content_batch = [ExtractText._read_image_for_api(path) for path in path_batch]
                
                image_content_batch = [item for item in image_content_batch if item is not None]

                if not image_content_batch:
                    pbar.update(len(path_batch))
                    continue

                api_results_batch = self._process_batch_with_google_api(image_content_batch)
                
                normalized_texts = [ExtractText._normalize_text(raw_text) for raw_text, _ in api_results_batch]

                for j, (_, path) in enumerate(api_results_batch):
                    text = normalized_texts[j]
                    if text:
                        # Giữ nguyên logic này vì path vẫn có cấu trúc thư mục cha là video_name
                        video_name = os.path.basename(os.path.dirname(path))
                        frame_name = os.path.splitext(os.path.basename(path))[0]
                        all_results[video_name][frame_name] = text
                
                pbar.update(len(path_batch))

        if not all_results:
            print("\nKhông trích xuất được văn bản nào.")
        else:
            print(f"\nTrích xuất hoàn tất. Đang lưu {len(all_results)} file JSON...")
            for video_name, video_data in all_results.items():
                output_json_path = os.path.join(self.output_path, f"{video_name}.json")
                try:
                    existing_data = {}
                    if os.path.exists(output_json_path):
                        with open(output_json_path, 'r', encoding='utf-8') as f:
                            existing_data = json.load(f)
                    
                    existing_data.update(video_data)

                    with open(output_json_path, 'w', encoding='utf-8') as f:
                        json.dump(existing_data, f, ensure_ascii=False, indent=4)
                    print(f" -> Đã lưu/cập nhật kết quả cho video {video_name} tại {output_json_path}")
                except Exception as e:
                    print(f"Lỗi khi lưu file cho video {video_name}: {e}")
        
        print("\n✅✅✅ Hoàn tất toàn bộ quá trình trích xuất! ✅✅✅")

In [5]:
extractor = ExtractText(
    input_path="/kaggle/input/kf-full", 
    output_path="/kaggle/working/ocr_data",
    progress={'length': 374251, 'start': 0, 'end': 16000}
)
extractor.extract_and_save()

Thư mục input: /kaggle/input/kf-full
Kết quả JSON sẽ được lưu tại: /kaggle/working/ocr_data
Đang quét và sắp xếp tất cả các đường dẫn ảnh...
{'length': 374251, 'start': 0, 'end': 16000}
Đã tìm thấy tổng cộng 374251 ảnh.
Sẽ xử lý 16000 ảnh từ chỉ số 0 đến 16000.


Đang trích xuất văn bản:   0%|          | 0/16000 [00:00<?, ?it/s]


Trích xuất hoàn tất. Đang lưu 31 file JSON...
 -> Đã lưu/cập nhật kết quả cho video L01_V001 tại /kaggle/working/ocr_data/L01_V001.json
 -> Đã lưu/cập nhật kết quả cho video L01_V002 tại /kaggle/working/ocr_data/L01_V002.json
 -> Đã lưu/cập nhật kết quả cho video L01_V003 tại /kaggle/working/ocr_data/L01_V003.json
 -> Đã lưu/cập nhật kết quả cho video L01_V004 tại /kaggle/working/ocr_data/L01_V004.json
 -> Đã lưu/cập nhật kết quả cho video L01_V005 tại /kaggle/working/ocr_data/L01_V005.json
 -> Đã lưu/cập nhật kết quả cho video L01_V006 tại /kaggle/working/ocr_data/L01_V006.json
 -> Đã lưu/cập nhật kết quả cho video L01_V007 tại /kaggle/working/ocr_data/L01_V007.json
 -> Đã lưu/cập nhật kết quả cho video L01_V008 tại /kaggle/working/ocr_data/L01_V008.json
 -> Đã lưu/cập nhật kết quả cho video L01_V009 tại /kaggle/working/ocr_data/L01_V009.json
 -> Đã lưu/cập nhật kết quả cho video L01_V010 tại /kaggle/working/ocr_data/L01_V010.json
 -> Đã lưu/cập nhật kết quả cho video L01_V011 tại /k